# P08 — PDF Extraction

Notebook ini mengekstrak teks per halaman dari satu PDF berbasis teks, memvalidasi file, memberi peringatan halaman kosong/kemungkinan scan, menampilkan preview, dan menyimpan hasil ke JSON. OCR tidak dilakukan pada notebook ini.

## 1. Instalasi dan import library

In [1]:
!pip -q install pymupdf

from pathlib import Path
import json
import re
import pymupdf
from google.colab import files

OUTPUT_DIR = Path("/content/data/processed_materials")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
MIN_TOTAL_CHARS = 20
MIN_PAGE_CHARS = 20


## 2. Upload satu file PDF

In [2]:
print("Pilih satu file PDF untuk diproses.")
uploaded = files.upload()

if not uploaded:
    raise RuntimeError("Tidak ada file yang diunggah. Jalankan ulang cell ini dan pilih PDF.")
if len(uploaded) != 1:
    raise ValueError("Unggah satu PDF saja dalam satu kali proses.")

filename, file_bytes = next(iter(uploaded.items()))
if not filename.lower().endswith(".pdf"):
    raise ValueError(f"File {filename!r} bukan PDF.")
if not file_bytes:
    raise ValueError("File PDF kosong (0 byte).")

print(f"File diterima: {filename} ({len(file_bytes)/1024:.2f} KB)")


Pilih satu file PDF untuk diproses.


Saving Alternifa - Memahami Informasi Artikel.pdf to Alternifa - Memahami Informasi Artikel.pdf
File diterima: Alternifa - Memahami Informasi Artikel.pdf (290.96 KB)


## 3. Buka dan validasi PDF

In [3]:
doc = None
try:
    doc = pymupdf.open(stream=file_bytes, filetype="pdf")
    if doc.is_encrypted:
        raise ValueError("PDF terenkripsi/terkunci password dan belum dapat diproses.")
    if len(doc) == 0:
        raise ValueError("PDF tidak memiliki halaman.")
except Exception as exc:
    if doc is not None:
        doc.close()
    raise RuntimeError(f"PDF tidak dapat dibuka: {exc}") from exc

print(f"PDF valid secara struktur: {len(doc)} halaman")


PDF valid secara struktur: 25 halaman


## 4. Ekstraksi teks per halaman

In [4]:
pages_payload = []
page_warnings = []
total_images = 0

for page_index, page in enumerate(doc):
    raw_text = page.get_text("text", sort=True) or ""
    text = raw_text.replace("\x00", " ")
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n{3,}", "\n\n", text).strip()
    image_count = len(page.get_images(full=True))
    total_images += image_count

    if len(text) < MIN_PAGE_CHARS:
        page_warnings.append({
            "page": page_index + 1,
            "text_characters": len(text),
            "image_count": image_count,
            "warning": "Teks sangat sedikit/kosong; periksa apakah halaman berupa scan atau gambar."
        })

    pages_payload.append({
        "page": page_index + 1,
        "text": text,
        "text_characters": len(text),
        "image_count": image_count
    })

full_text = "\n".join(item["text"] for item in pages_payload).strip()
if len(full_text) < MIN_TOTAL_CHARS:
    doc.close()
    raise ValueError(
        f"Teks yang berhasil diekstrak hanya {len(full_text)} karakter. "
        "PDF mungkin hasil scan atau tidak memiliki text layer. Notebook ini belum menjalankan OCR."
    )

metadata = doc.metadata or {}
summary = {
    "source_file": filename,
    "page_count": len(doc),
    "file_size_bytes": len(file_bytes),
    "total_extracted_characters": len(full_text),
    "total_images": total_images,
    "metadata": metadata,
    "pages_with_warnings": page_warnings
}
doc.close()

print(f"Berhasil mengekstrak {len(full_text):,} karakter dari {len(pages_payload)} halaman.")
print(f"Halaman yang perlu diperiksa: {len(page_warnings)}")


Berhasil mengekstrak 63,162 karakter dari 25 halaman.
Halaman yang perlu diperiksa: 0


## 5. Tampilkan ringkasan dan preview

In [5]:
display(summary)
for item in pages_payload[:3]:
    print(f"\n--- HALAMAN {item['page']} | {item['text_characters']} karakter ---")
    print(item["text"][:1500] if item["text"] else "[Tidak ada teks yang terdeteksi]")

if page_warnings:
    print("\nPERINGATAN HALAMAN:")
    display(page_warnings)


{'source_file': 'Alternifa - Memahami Informasi Artikel.pdf',
 'page_count': 25,
 'file_size_bytes': 297942,
 'total_extracted_characters': 63162,
 'total_images': 25,
 'metadata': {'format': 'PDF 1.7',
  'title': '',
  'author': 'Imam Maulana',
  'subject': '',
  'keywords': '',
  'creator': 'Microsoft® Word untuk Microsoft 365',
  'producer': 'Microsoft® Word untuk Microsoft 365',
  'creationDate': "D:20250325035739+07'00'",
  'modDate': "D:20250325035739+07'00'",
  'trapped': '',
  'encryption': None},
 'pages_with_warnings': []}


--- HALAMAN 1 | 2752 karakter ---
Pulang Mudik: Selebrasi Agama dan Budaya

Acara pulang mudik Lebaran selalu disambut dengan antusias oleh masyarakat layaknya
sebuah selebrasi kolosal. Pulang mudik bukan sekadar melepas rindu kampung halaman,
melainkan juga dihayati sebagai rangkaian rasa syukur dengan berakhirnya ibadah puasa
selama sebulan.

Ada keyakinan bagi mereka yang berpuasa bahwa ampunan Tuhan yang dijanjikan dalam
bulan Ramadhan akan terganjal jika antara anak dan orangtua dan antara sesama teman serta
keluarga belum saling memaafkan. Makanya acara pulang mudik juga punya makna spiritual
yang kental, di samping melepas rindu dan napak tilas dengan kampung halaman. Mereka
berkumpul dan berhalalbihalal saling memaafkan terutama dengan keluarga dan tetangga.

Selama bulan Ramadhan umat Islam bagaikan berjalan memasuki lorong waktu yang
suasananya serba bernuansa spiritual. Tiba-tiba mereka lebih dekat dengan Tuhan, dekat
dengan masjid, dengan kitab suci, rajin sembahyang, raji

## 6. Simpan hasil ekstraksi

In [6]:
safe_stem = re.sub(r"[^A-Za-z0-9._-]+", "_", Path(filename).stem).strip("._") or "materi"
output_file = OUTPUT_DIR / f"{safe_stem}_extracted_pages.json"
output_payload = {
    "metadata": summary,
    "pages": [{"page": p["page"], "text": p["text"]} for p in pages_payload]
}
with output_file.open("w", encoding="utf-8") as f:
    json.dump(output_payload, f, ensure_ascii=False, indent=2)

print(f"Output tersimpan: {output_file}")
files.download(str(output_file))


Output tersimpan: /content/data/processed_materials/Alternifa_-_Memahami_Informasi_Artikel_extracted_pages.json


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## 7. Catatan interpretasi

- Halaman dengan teks sedikit tidak otomatis pasti scan; periksa preview dan PDF aslinya.
- Jika PDF berupa scan, OCR merupakan pekerjaan tambahan dan belum termasuk notebook ini.
- `pages` mempertahankan nomor halaman agar P09 dapat melacak asal teks.
- File JSON ini menjadi input notebook P09.